# WeaveGuard AI — Audit du modèle YOLO sur le dataset v2

**Objectifs**
1. Vérifier la cohérence entre `data.yaml`, les noms de classes, les annotations et le modèle exporté.
2. Auditer les images et boîtes de la classe `Hole` (tailles, dimensions, exemples).
3. Générer une galerie de faux positifs et faux négatifs sur le test.
4. Tester plusieurs seuils de confiance **sur la validation uniquement** et calculer des métriques par classe.
5. Exporter les tableaux et les figures dans un dossier de résultats.

> **Important :** le jeu de test sert uniquement à l'analyse finale des erreurs. Le seuil est choisi sur `val`, jamais sur `test`. Ce notebook ne réentraîne pas le modèle et ne modifie pas le dataset.


## 0. Environnement

Exécuter dans Google Colab avec le même environnement que l'entraînement. Monter Google Drive si les fichiers sont stockés dessus.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip -q install ultralytics pandas pyyaml seaborn scikit-learn opencv-python-headless


## 1. Chemins et paramètres

Vérifie ces chemins avant de lancer les cellules. Le chemin du modèle doit pointer vers un fichier `.pt` réel, par exemple `best.pt`, et non uniquement vers un dossier d'export.


In [ ]:
from pathlib import Path
import os, json, yaml, math, random, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from IPython.display import display

# Dataset v2 indiqué dans les résultats précédents
DATA_YAML = Path('/content/weaveguard_dataset_v2/data.yaml')

# À ADAPTER : renseigner le vrai fichier de poids .pt.
# Exemples possibles :
MODEL_PATH = Path('/content/drive/MyDrive/WeaveGuard/models/yolo/v8n_640/weights/best.pt')
# MODEL_PATH = Path('/content/drive/MyDrive/WeaveGuard/models/weaveguard_yolo_v1/best.pt')

OUTPUT_DIR = Path('/content/drive/MyDrive/WeaveGuard/audit_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMGSZ = 640
IOU_NMS = 0.6
CONF_DISPLAY = 0.10
CONF_SWEEP = np.round(np.arange(0.05, 0.801, 0.05), 2).tolist()
MAX_GALLERY = 24
SEED = 42

print('data.yaml existe :', DATA_YAML.exists(), DATA_YAML)
print('Modèle existe :', MODEL_PATH.exists(), MODEL_PATH)
print('Dossier de sortie :', OUTPUT_DIR)


In [ ]:
assert DATA_YAML.exists(), f'Fichier data.yaml introuvable : {DATA_YAML}. Modifie DATA_YAML.'
assert MODEL_PATH.exists(), f'Fichier de poids .pt introuvable : {MODEL_PATH}. Modifie MODEL_PATH.'


## 2. Vérifier les classes et les chemins du dataset

Cette cellule compare les classes déclarées dans `data.yaml` avec les statistiques connues du v2. Elle affiche aussi les chemins résolus pour `train`, `val` et `test`.


In [ ]:
with open(DATA_YAML, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

print(json.dumps(data_cfg, indent=2, ensure_ascii=False))
names_cfg = data_cfg.get('names')
if isinstance(names_cfg, dict):
    class_names = [names_cfg[k] for k in sorted(names_cfg, key=lambda x: int(x))]
elif isinstance(names_cfg, list):
    class_names = names_cfg
else:
    raise ValueError("Champ 'names' absent ou format inconnu dans data.yaml")

print('\nNombre de classes :', len(class_names))
for i, n in enumerate(class_names):
    print(i, '->', n)

expected = ['DropStitch', 'Hole', 'Horizontal Mark', 'Knot', 'Oil Spot',
            'Setup', 'Slub', 'Vertical LineMark', 'Yurn Dust']
print('\nClasses attendues présentes dans le rapport :', class_names == expected)
if class_names != expected:
    print('ATTENTION : vérifier le mapping avec les labels et le rapport d’entraînement.')


In [ ]:
from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))
model_names = model.names
if isinstance(model_names, dict):
    model_class_names = [model_names[k] for k in sorted(model_names)]
else:
    model_class_names = list(model_names)

print('Classes du modèle :')
for i, n in enumerate(model_class_names):
    print(i, '->', n)

mapping_ok = (class_names == model_class_names)
print('\nMapping data.yaml / modèle identique :', mapping_ok)
if not mapping_ok:
    print('⚠️ STOP : ne pas interpréter les métriques avant d’avoir résolu le mapping des classes.')


## 3. Résoudre les dossiers et auditer les annotations YOLO

On vérifie l'existence des images et labels, les lignes mal formées, les classes hors limites, les coordonnées hors plage et les boîtes de largeur/hauteur nulles. Les images sans label peuvent être normales dans certains datasets, donc elles sont rapportées sans être automatiquement déclarées invalides.


In [ ]:
from pathlib import Path

def resolve_dataset_path(p):
    p = Path(str(p))
    if p.is_absolute() and p.exists():
        return p
    # Ultralytics peut stocker un chemin racine dans data.yaml
    root = Path(data_cfg.get('path', DATA_YAML.parent))
    candidates = [root / p, DATA_YAML.parent / p, Path('/content') / p]
    for c in candidates:
        if c.exists():
            return c.resolve()
    return candidates[0].resolve()

split_paths = {s: resolve_dataset_path(data_cfg[s]) for s in ['train', 'val', 'test'] if s in data_cfg}
split_paths


In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

def images_from_split(path):
    path = Path(path)
    if path.is_dir():
        return sorted([p for p in path.rglob('*') if p.suffix.lower() in IMG_EXTS])
    if path.is_file() and path.suffix.lower() == '.txt':
        rows = [x.strip() for x in path.read_text(encoding='utf-8').splitlines() if x.strip()]
        return [Path(x) if Path(x).is_absolute() else (path.parent / x).resolve() for x in rows]
    raise FileNotFoundError(f'Chemin de split non reconnu : {path}')

def label_for_image(img_path):
    # Convention YOLO courante : .../images/... -> .../labels/... et .txt
    parts = list(img_path.parts)
    if 'images' in parts:
        i = len(parts) - 1 - parts[::-1].index('images')
        parts[i] = 'labels'
        return Path(*parts).with_suffix('.txt')
    # Sinon, recherche dans le dossier labels voisin de la racine du split si possible
    return img_path.with_suffix('.txt')

annotation_rows = []
split_image_lists = {}
for split, p in split_paths.items():
    imgs = images_from_split(p)
    split_image_lists[split] = imgs
    missing_labels = 0
    for im in imgs:
        lab = label_for_image(im)
        if not lab.exists():
            missing_labels += 1
            continue
        try:
            lines = [x.strip() for x in lab.read_text(encoding='utf-8').splitlines() if x.strip()]
        except Exception:
            annotation_rows.append({'split': split, 'image': str(im), 'label': str(lab),
                                    'issue': 'label_unreadable', 'line': '', 'class_id': None})
            continue
        for line_no, line in enumerate(lines, start=1):
            vals = line.split()
            issue = None
            cls_id = None
            if len(vals) != 5:
                issue = 'wrong_number_of_fields'
            else:
                try:
                    cls_f = float(vals[0])
                    coords = [float(v) for v in vals[1:]]
                    cls_id = int(cls_f)
                    if cls_f != cls_id:
                        issue = 'class_id_not_integer'
                    elif not (0 <= cls_id < len(class_names)):
                        issue = 'class_id_out_of_range'
                    elif not all(np.isfinite(coords)):
                        issue = 'non_finite_coordinate'
                    elif not all(0 <= c <= 1 for c in coords):
                        issue = 'coordinate_out_of_range'
                    elif coords[2] <= 0 or coords[3] <= 0:
                        issue = 'zero_or_negative_box_size'
                except Exception:
                    issue = 'non_numeric_value'
            if issue:
                annotation_rows.append({'split': split, 'image': str(im), 'label': str(lab),
                                        'issue': issue, 'line': line_no, 'class_id': cls_id})
    print(f'{split}: {len(imgs)} images ; labels manquants/non trouvés : {missing_labels}')

annotation_issues = pd.DataFrame(annotation_rows)
print('\nProblèmes d’annotation détectés :', len(annotation_issues))
display(annotation_issues.head(30))
annotation_issues.to_csv(OUTPUT_DIR / 'annotation_issues.csv', index=False)


## 4. Distribution des boîtes et audit de la classe Hole

La taille normalisée d'une boîte est calculée à partir de `w × h`. Les dimensions en pixels sont estimées à partir de la résolution de l'image. Cela permet d'identifier si les trous sont souvent très petits, mais ne remplace pas une inspection visuelle.


In [ ]:
box_rows = []
for split, imgs in split_image_lists.items():
    for im in imgs:
        lab = label_for_image(im)
        if not lab.exists():
            continue
        try:
            with Image.open(im) as image:
                W, H = image.size
            lines = [x.strip() for x in lab.read_text(encoding='utf-8').splitlines() if x.strip()]
            for line in lines:
                vals = line.split()
                if len(vals) != 5:
                    continue
                cls_id = int(float(vals[0]))
                xc, yc, bw, bh = map(float, vals[1:])
                if not (0 <= cls_id < len(class_names)):
                    continue
                box_rows.append({
                    'split': split, 'image': str(im), 'label': str(lab),
                    'class_id': cls_id, 'class_name': class_names[cls_id],
                    'image_w': W, 'image_h': H,
                    'box_w_norm': bw, 'box_h_norm': bh,
                    'box_w_px': bw * W, 'box_h_px': bh * H,
                    'box_area_norm': bw * bh, 'box_area_px': bw * bh * W * H
                })
        except Exception as e:
            print('Lecture ignorée :', im, e)

boxes_df = pd.DataFrame(box_rows)
display(boxes_df.groupby(['split','class_name']).size().unstack(fill_value=0))
boxes_df.to_csv(OUTPUT_DIR / 'all_boxes.csv', index=False)

hole_df = boxes_df[boxes_df['class_name'].str.lower().eq('hole')].copy()
print('Nombre total de boîtes Hole :', len(hole_df))
display(hole_df.groupby('split')[['box_w_px','box_h_px','box_area_px']].describe().round(2))

if not hole_df.empty:
    fig, ax = plt.subplots(figsize=(9, 4))
    for split in ['train', 'val', 'test']:
        vals = hole_df.loc[hole_df.split == split, 'box_area_px']
        if len(vals):
            ax.hist(np.log10(vals.clip(lower=1)), bins=20, alpha=0.45, label=split)
    ax.set_title('Hole — distribution de log10 de la surface des boîtes (pixels²)')
    ax.set_xlabel('log10(surface en pixels²)')
    ax.set_ylabel('Nombre de boîtes')
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'hole_box_area_distribution.png', dpi=160)
    plt.show()


## 5. Visualiser des exemples annotés Hole

Cette cellule affiche jusqu'à 24 images avec les boîtes de vérité terrain, en privilégiant les plus petites boîtes `Hole`. Cela permet de vérifier les annotations et de voir la difficulté visuelle réelle des micro-trous.


In [ ]:
def load_yolo_boxes(label_path, W, H):
    out = []
    if not label_path.exists():
        return out
    for line in label_path.read_text(encoding='utf-8').splitlines():
        vals = line.strip().split()
        if len(vals) != 5:
            continue
        try:
            cid = int(float(vals[0]))
            xc, yc, bw, bh = map(float, vals[1:])
            x1 = (xc - bw/2) * W; y1 = (yc - bh/2) * H
            x2 = (xc + bw/2) * W; y2 = (yc + bh/2) * H
            out.append((cid, x1, y1, x2, y2))
        except Exception:
            pass
    return out

hole_images = []
for _, r in hole_df.sort_values('box_area_px').iterrows():
    if r['split'] not in ['train', 'val', 'test']:
        continue
    p = Path(r['image'])
    if p not in [x[0] for x in hole_images]:
        hole_images.append((p, r['split'], r['box_area_px']))
hole_images = hole_images[:MAX_GALLERY]

if hole_images:
    ncols = 4
    nrows = math.ceil(len(hole_images) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4*nrows))
    axes = np.array(axes).reshape(-1)
    for ax, (im_path, split, area) in zip(axes, hole_images):
        im = Image.open(im_path).convert('RGB')
        W, H = im.size
        draw = ImageDraw.Draw(im)
        for cid, x1, y1, x2, y2 in load_yolo_boxes(label_for_image(im_path), W, H):
            if 0 <= cid < len(class_names):
                draw.rectangle((x1,y1,x2,y2), outline='red' if class_names[cid].lower() == 'hole' else 'yellow', width=3)
                draw.text((x1, max(0,y1-14)), class_names[cid], fill='red' if class_names[cid].lower() == 'hole' else 'yellow')
        ax.imshow(im)
        ax.set_title(f'{split} | {im_path.name}\nHole area ≈ {area:.0f} px²')
        ax.axis('off')
    for ax in axes[len(hole_images):]:
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'hole_ground_truth_gallery.png', dpi=160)
    plt.show()
else:
    print('Aucune boîte Hole trouvée : vérifier le chemin des labels et les noms de classes.')


## 6. Balayer les seuils de confiance sur la validation

Le seuil global 0,3113 issu de l'expérience précédente est seulement un point de comparaison. Ici, nous évaluons différents seuils sur `val`. La métrique de rappel est calculée en associant chaque prédiction à une vérité terrain de la même classe avec IoU ≥ 0,50, au maximum une fois par boîte annotée. Les boîtes de vérité terrain sans prédiction correspondante sont des faux négatifs. Les prédictions restantes sont des faux positifs.

Cette implémentation simplifiée sert au diagnostic des seuils ; les métriques mAP officielles restent celles d'Ultralytics.


In [ ]:
def iou_xyxy(a, b):
    xA = max(a[0], b[0]); yA = max(a[1], b[1])
    xB = min(a[2], b[2]); yB = min(a[3], b[3])
    inter = max(0, xB-xA) * max(0, yB-yA)
    area_a = max(0,a[2]-a[0]) * max(0,a[3]-a[1])
    area_b = max(0,b[2]-b[0]) * max(0,b[3]-b[1])
    return inter / (area_a + area_b - inter + 1e-9)

def gt_for_image(img_path):
    with Image.open(img_path) as im:
        W, H = im.size
    return W, H, load_yolo_boxes(label_for_image(img_path), W, H)

def score_predictions(split, conf, iou_match=0.5, max_images=None):
    imgs = split_image_lists[split]
    if max_images:
        imgs = imgs[:max_images]
    counts = {i: {'tp':0, 'fp':0, 'fn':0} for i in range(len(class_names))}
    for start in range(0, len(imgs), 16):
        batch = imgs[start:start+16]
        results = model.predict([str(p) for p in batch], imgsz=IMGSZ, conf=conf,
                                iou=IOU_NMS, verbose=False, device=None)
        for img_path, result in zip(batch, results):
            W, H, gt_all = gt_for_image(img_path)
            pred_all = []
            if result.boxes is not None:
                for b in result.boxes:
                    cid = int(b.cls.item())
                    xyxy = b.xyxy[0].cpu().numpy().tolist()
                    pred_all.append({'cid':cid, 'box':xyxy, 'conf':float(b.conf.item())})
            for cid in range(len(class_names)):
                gts = [g[1:] for g in gt_all if g[0] == cid]
                preds = [p for p in pred_all if p['cid'] == cid]
                matched_gt = set()
                # Trier par confiance décroissante pour une association gloutonne déterministe
                preds = sorted(preds, key=lambda p: p['conf'], reverse=True)
                tp = 0
                for pred in preds:
                    best_iou, best_j = 0, None
                    for j, gt in enumerate(gts):
                        if j in matched_gt:
                            continue
                        score = iou_xyxy(pred['box'], gt)
                        if score > best_iou:
                            best_iou, best_j = score, j
                    if best_j is not None and best_iou >= iou_match:
                        tp += 1
                        matched_gt.add(best_j)
                    else:
                        counts[cid]['fp'] += 1
                counts[cid]['tp'] += tp
                counts[cid]['fn'] += len(gts) - tp
    rows = []
    for cid, c in counts.items():
        p = c['tp'] / (c['tp'] + c['fp']) if c['tp'] + c['fp'] else 0.0
        r = c['tp'] / (c['tp'] + c['fn']) if c['tp'] + c['fn'] else 0.0
        f1 = 2*p*r/(p+r) if p+r else 0.0
        rows.append({'split':split, 'conf':conf, 'class_id':cid, 'class_name':class_names[cid],
                     'TP':c['tp'], 'FP':c['fp'], 'FN':c['fn'],
                     'precision':p, 'recall':r, 'F1':f1})
    return rows

val_sweep_rows = []
for conf in CONF_SWEEP:
    print(f'Évaluation val conf={conf:.2f}')
    val_sweep_rows.extend(score_predictions('val', conf, iou_match=0.5))

val_sweep = pd.DataFrame(val_sweep_rows)
val_sweep.to_csv(OUTPUT_DIR / 'threshold_sweep_val.csv', index=False)
display(val_sweep[val_sweep.class_name.str.lower().eq('hole')].round(3))


In [ ]:
# Résumé par classe : meilleur F1 sur validation et seuil correspondant
best_by_class = (val_sweep.sort_values('F1', ascending=False)
                 .groupby('class_name', as_index=False).first()
                 .sort_values('class_name'))
display(best_by_class[['class_name','conf','precision','recall','F1','TP','FP','FN']].round(3))

# Courbes precision/recall/F1 pour Hole
hole_sweep = val_sweep[val_sweep.class_name.str.lower().eq('hole')].sort_values('conf')
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(hole_sweep.conf, hole_sweep.precision, marker='o', label='Précision')
ax.plot(hole_sweep.conf, hole_sweep.recall, marker='o', label='Rappel')
ax.plot(hole_sweep.conf, hole_sweep.F1, marker='o', label='F1')
ax.axvline(0.3113, linestyle='--', label='Seuil précédent 0,3113')
ax.set_xlabel('Seuil de confiance')
ax.set_ylabel('Score')
ax.set_ylim(0,1.02)
ax.set_title('Hole — seuils évalués sur validation')
ax.grid(True, alpha=0.25)
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'hole_threshold_sweep_val.png', dpi=160)
plt.show()


## 7. Choisir un seuil global de diagnostic sur validation

Cette cellule sélectionne le seuil qui maximise le **F1 macro** sur les classes présentes. Ce choix est un outil de comparaison, pas forcément le seuil optimal pour la sécurité industrielle. Pour un usage où manquer un trou est plus coûteux qu'une fausse alerte, il peut être préférable de choisir un seuil plus bas afin d'augmenter le rappel, puis de mesurer le coût en faux positifs.


In [ ]:
macro_by_conf = (val_sweep.groupby('conf')
                 .agg(macro_precision=('precision','mean'),
                      macro_recall=('recall','mean'),
                      macro_F1=('F1','mean'))
                 .reset_index())
display(macro_by_conf.sort_values('macro_F1', ascending=False).head(10).round(3))

best_macro = macro_by_conf.loc[macro_by_conf['macro_F1'].idxmax()]
print('Seuil maximisant le F1 macro simplifié sur val :', float(best_macro['conf']))
print('F1 macro :', float(best_macro['macro_F1']))
print('Rappel macro :', float(best_macro['macro_recall']))


## 8. Générer une galerie des erreurs sur test

Cette cellule utilise un seuil fixé (par défaut 0,3113) pour visualiser :
- **Faux négatif (FN)** : une boîte annotée sans prédiction de même classe avec IoU ≥ 0,50.
- **Faux positif (FP)** : une prédiction qui ne correspond à aucune boîte annotée de la même classe.
- **Trou détecté** : prédiction `Hole` correctement associée.

Le test est réservé à l'analyse finale des erreurs. Ne choisissez pas le seuil en fonction des résultats test.


In [ ]:
def draw_diagnostic_image(img_path, result, conf, iou_match=0.5):
    im = Image.open(img_path).convert('RGB')
    W, H = im.size
    draw = ImageDraw.Draw(im)
    gt = load_yolo_boxes(label_for_image(img_path), W, H)
    preds = []
    if result.boxes is not None:
        for b in result.boxes:
            cid = int(b.cls.item())
            preds.append({'cid':cid, 'box':b.xyxy[0].cpu().numpy().tolist(), 'conf':float(b.conf.item())})
    matched_pred = set()
    matched_gt = set()
    # Matching par classe
    for cid in range(len(class_names)):
        gi = [j for j,g in enumerate(gt) if g[0] == cid]
        pi = [j for j,p in enumerate(preds) if p['cid'] == cid]
        for pj in sorted(pi, key=lambda j: preds[j]['conf'], reverse=True):
            best = (0, None)
            for gj in gi:
                if gj in matched_gt:
                    continue
                iou = iou_xyxy(preds[pj]['box'], gt[gj][1:])
                if iou > best[0]:
                    best = (iou, gj)
            if best[1] is not None and best[0] >= iou_match:
                matched_pred.add(pj); matched_gt.add(best[1])
    # GT : vert si match, rouge si FN
    for j, g in enumerate(gt):
        cid, x1,y1,x2,y2 = g
        color = 'lime' if j in matched_gt else 'red'
        label = f'GT {class_names[cid]}' + ('' if j in matched_gt else ' [FN]')
        draw.rectangle((x1,y1,x2,y2), outline=color, width=3)
        draw.text((x1, max(0,y1-14)), label, fill=color)
    # Predictions : cyan si match, magenta si FP
    for j,p in enumerate(preds):
        x1,y1,x2,y2 = p['box']; cid=p['cid']
        color = 'cyan' if j in matched_pred else 'magenta'
        label = f'P {class_names[cid]} {p["conf"]:.2f}' + ('' if j in matched_pred else ' [FP]')
        draw.rectangle((x1,y1,x2,y2), outline=color, width=2)
        draw.text((x1, min(H-14,y2+2)), label, fill=color)
    return im, gt, preds, matched_gt, matched_pred

TEST_CONF = 0.3113113113
test_imgs = split_image_lists['test']
test_results = []
test_error_rows = []
for start in range(0, len(test_imgs), 16):
    batch = test_imgs[start:start+16]
    results = model.predict([str(p) for p in batch], imgsz=IMGSZ, conf=TEST_CONF,
                            iou=IOU_NMS, verbose=False)
    for pth, res in zip(batch, results):
        test_results.append((pth, res))
        _, gt, preds, mg, mp = draw_diagnostic_image(pth, res, TEST_CONF)
        for j,g in enumerate(gt):
            if j not in mg:
                test_error_rows.append({'image':str(pth), 'error':'FN', 'class_name':class_names[g[0]], 'conf':None})
        for j,p in enumerate(preds):
            if j not in mp:
                test_error_rows.append({'image':str(pth), 'error':'FP', 'class_name':class_names[p['cid']], 'conf':p['conf']})

test_errors = pd.DataFrame(test_error_rows)
test_errors.to_csv(OUTPUT_DIR / 'test_error_inventory.csv', index=False)
display(test_errors.groupby(['error','class_name']).size().unstack(fill_value=0))


In [ ]:
# Galerie des faux négatifs Hole d'abord, puis autres erreurs
fn_hole_paths = set(test_errors[(test_errors.error=='FN') & (test_errors.class_name.str.lower()=='hole')]['image'].tolist())
fp_hole_paths = set(test_errors[(test_errors.error=='FP') & (test_errors.class_name.str.lower()=='hole')]['image'].tolist())
other_error_paths = set(test_errors['image'].tolist())
priority_paths = list(fn_hole_paths) + [p for p in fp_hole_paths if p not in fn_hole_paths]
priority_paths += [p for p in other_error_paths if p not in set(priority_paths)]
priority_paths = priority_paths[:MAX_GALLERY]

result_by_path = {str(p):r for p,r in test_results}
if priority_paths:
    ncols=3
    nrows=math.ceil(len(priority_paths)/ncols)
    fig, axes = plt.subplots(nrows,ncols,figsize=(18,6*nrows))
    axes=np.array(axes).reshape(-1)
    for ax, pstr in zip(axes, priority_paths):
        pth=Path(pstr)
        im, *_ = draw_diagnostic_image(pth, result_by_path[pstr], TEST_CONF)
        ax.imshow(im)
        tags=[]
        if pstr in fn_hole_paths: tags.append('FN Hole')
        if pstr in fp_hole_paths: tags.append('FP Hole')
        ax.set_title(f'{pth.name}\n{", ".join(tags) if tags else "Autre erreur"}')
        ax.axis('off')
    for ax in axes[len(priority_paths):]:
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'test_error_gallery.png', dpi=150)
    plt.show()
else:
    print('Aucune erreur détectée par cette procédure. Vérifier le mapping et les chemins d’annotations.')


## 9. Vérification finale : rapport de synthèse

Le rapport regroupe les paramètres utilisés, les classes, les effectifs réellement trouvés et les fichiers produits. Conserver ce rapport avec le checkpoint et le hash du dataset v2.


In [ ]:
summary = {
    'dataset_yaml': str(DATA_YAML),
    'model_path': str(MODEL_PATH),
    'imgsz': IMGSZ,
    'iou_nms': IOU_NMS,
    'test_conf_for_error_gallery': TEST_CONF,
    'classes_data_yaml': class_names,
    'classes_model': model_class_names,
    'mapping_ok': bool(mapping_ok),
    'images_found_per_split': {s:len(v) for s,v in split_image_lists.items()},
    'annotation_issues_count': int(len(annotation_issues)),
    'boxes_per_class_split': boxes_df.groupby(['split','class_name']).size().unstack(fill_value=0).to_dict() if not boxes_df.empty else {},
    'outputs': [str(p) for p in sorted(OUTPUT_DIR.glob('*'))]
}
with open(OUTPUT_DIR / 'audit_summary.json','w',encoding='utf-8') as f:
    json.dump(summary,f,indent=2,ensure_ascii=False,default=str)
print(json.dumps(summary,indent=2,ensure_ascii=False,default=str))
print('\nRésultats enregistrés dans :', OUTPUT_DIR)


## Interprétation à faire après exécution

- Si `mapping_ok` vaut `False`, arrêter l'analyse et corriger le mapping avant toute conclusion.
- Si `annotation_issues_count` est non nul, examiner `annotation_issues.csv`.
- Regarder `hole_ground_truth_gallery.png` pour juger la taille et la qualité des annotations de trous.
- Regarder `hole_threshold_sweep_val.png` et `threshold_sweep_val.csv` pour choisir un seuil sur validation.
- Examiner `test_error_gallery.png` et `test_error_inventory.csv` pour comprendre les erreurs finales, sans ajuster les paramètres à partir du test.
- Les métriques de seuil calculées ici sont une procédure de diagnostic indépendante ; pour une comparaison rigoureuse, conserver également les métriques officielles Ultralytics et les mêmes paramètres d'évaluation.
